# Chat SSE Streaming Test

This notebook tests the `/api/chat` endpoint to verify that **Server-Sent Events** are streamed incrementally (not batched).

> **Pre-requisite**: The FastAPI server must be running:  
> `uvicorn fastapi_app.app:app --reload --host 127.0.0.1 --port 8000`


In [1]:
import httpx
import time
import json

BASE_URL = "http://127.0.0.1:8000"
CHAT_URL = f"{BASE_URL}/api/chat"

## 1. Health Check

Quick sanity check that the server is reachable.


In [9]:
r = httpx.get(f"{BASE_URL}/health")
print(r.status_code, r.json())

200 {'status': 'ok'}


## 2. Stream a Chat Message

Send a question and print each SSE event **as it arrives**, along with the wall-clock time since the request started.  
If streaming works correctly you'll see timestamps spread across several seconds, not all at once.


In [10]:
def parse_sse_events(text: str):
    """Parse an SSE text block into a list of (event_type, data_dict) tuples."""
    events = []
    event_type = ""
    for line in text.splitlines():
        if line.startswith("event: "):
            event_type = line[7:]
        elif line.startswith("data: "):
            try:
                data = json.loads(line[6:])
            except json.JSONDecodeError:
                data = line[6:]
            events.append((event_type, data))
            event_type = ""
    return events

In [11]:
# payload = {
#     "message": "List the tables in the database",
#     "session_id": "notebook-test",
# }
payload = {
    "message": "Does money spent on meta ads aid the revenue. Do a deep analysis. **Don't create charts.**",
    "session_id": "notebook-test",
}


print("Streaming response...\n")
start = time.perf_counter()
token_count = 0
first_token_time = None

with httpx.stream("POST", CHAT_URL, json=payload, timeout=120) as resp:
    buffer = ""
    for chunk in resp.iter_text():
        elapsed = time.perf_counter() - start
        buffer += chunk

        # Process complete SSE frames (separated by double newline)
        while "\n\n" in buffer:
            frame, buffer = buffer.split("\n\n", 1)
            for event_type, data in parse_sse_events(frame):
                if event_type == "token":
                    token_count += 1
                    if first_token_time is None:
                        first_token_time = elapsed
                    print(f"  [{elapsed:6.2f}s] token: {data['content']!r}")
                elif event_type == "tool_call_start":
                    print(
                        f"  [{elapsed:6.2f}s] 🔧 TOOL START: {data['tool_name']}"
                    )
                elif event_type == "tool_result":
                    result_preview = str(data["result"])[:120]
                    print(
                        f"  [{elapsed:6.2f}s] ✅ TOOL RESULT ({data['tool_name']}): {result_preview}"
                    )
                elif event_type == "chart":
                    print(
                        f"  [{elapsed:6.2f}s] 📊 CHART: {data['chart_type']}"
                    )
                elif event_type == "end":
                    print(f"  [{elapsed:6.2f}s] ⏹ END")
                elif event_type == "error":
                    print(f"  [{elapsed:6.2f}s] ❌ ERROR: {data['detail']}")

total = time.perf_counter() - start
print(f"\n--- Summary ---")
print(f"Total tokens received : {token_count}")
print(
    f"Time to first token   : {first_token_time:.2f}s"
    if first_token_time
    else "No tokens received"
)
print(f"Total elapsed time    : {total:.2f}s")
print(
    f"Streaming works       : {'✅ YES' if first_token_time and total - first_token_time > 0.3 else '❌ NO (tokens arrived in a single burst)'}"
)

Streaming response...

  [  1.75s] 🔧 TOOL START: sql_db_schema
  [  1.76s] 🔧 TOOL START: sql_db_schema
  [  1.78s] ✅ TOOL RESULT (sql_db_schema): /* Column descriptions for meta_ad_insights:
   - campaign_name: Name of the parent campaign this ad belongs to
   - ads
  [  1.78s] ✅ TOOL RESULT (sql_db_schema): /* Column descriptions for meta_daily_insights:
   - date: Date of the daily aggregated metrics (YYYY-MM-DD format)
   -
  [  2.27s] 🔧 TOOL START: sql_db_query
  [  2.72s] ✅ TOOL RESULT (sql_db_query): [{'date': '2026-02-23', 'spend': 2279.12, 'revenue': 4095.0, 'roas': 1.796746}, {'date': '2026-02-22', 'spend': 7478.16,
  [  5.55s] 🔧 TOOL START: sql_db_query
  [  5.62s] 🔧 TOOL START: sql_db_query
  [  5.67s] 🔧 TOOL START: sql_db_query
  [  5.68s] 🔧 TOOL START: sql_db_query
  [  5.70s] 🔧 TOOL START: sql_db_query
  [  5.74s] ✅ TOOL RESULT (sql_db_query): [{'avg_roas': 3.0305190925925927, 'avg_spend': 4608.780925925927, 'avg_revenue': 13399.507407407407, 'total_spend': 2488
  [  5.74

## 3. Test Conversation History

After the streamed chat above, verify that the conversation was persisted.


In [5]:
r = httpx.get(f"{BASE_URL}/api/chat/history/notebook-test")
history = r.json()
print(f"Messages in history: {len(history)}\n")
for msg in history:
    role = msg["role"]
    content = (msg.get("content") or "")[:120]
    extra = ""
    if msg.get("tool_calls"):
        names = [tc["name"] for tc in msg["tool_calls"]]
        extra = f"  [tools: {', '.join(names)}]"
    if msg.get("tool_name"):
        extra = f"  [tool: {msg['tool_name']}]"
    print(f"  {role:>10}: {content}{extra}")

Messages in history: 4

        user: List the tables in the database
   assistant:   [tools: sql_db_list_tables]
        tool: meta_ad_insights, meta_adset_insights, meta_campaign_insights, meta_daily_insights, shopify_orders  [tool: sql_db_list_tables]
   assistant: The tables in the database are:

1. meta_ad_insights
2. meta_adset_insights
3. meta_campaign_insights
4. meta_daily_insi


## 4. Cleanup — Delete Test Session


In [6]:
r = httpx.delete(f"{BASE_URL}/api/chat/history/notebook-test")
print(r.status_code, r.json())

200 {'status': 'deleted', 'session_id': 'notebook-test'}
